# Lab 02-3. Similarity Graph Construction

# Overview

In this lab, we use **Iris** and **Moons** to examine three questions:

1. How does an $\epsilon$-graph decide which pairs become edges?
2. How does a kNN graph select neighbors, and why can a node's degree exceed $k$?
3. How do heat-kernel weights decay as Euclidean distance grows?

> #### 📝 Implement in `lab02_3.py` first
>
> This notebook calls functions from `lab02_3.py`. Find each
> `# ========== TODO ==========` block, remove `raise NotImplementedError`,
> and write your implementation. Restart the kernel after editing the `.py`
> file, then run this notebook from the top.
>
> On the course site, Practice cell outputs are the expected results after
> those functions are implemented. Your local notebook will not produce them
> until the TODOs are done.
>
> The same functions are reused on the Moons data in the last section, so
> implement them before that part.
>
> Check your functions with:
>
> ```bash
> python -m doctest lab02_3.py -v
> ```

In [ ]:
#| label: setup-similarity-graph
#| include: false

from pathlib import Path
import sys

_lab = Path("exercises/lab02")
if not (_lab / "helper.py").exists():
    _lab = Path(".")
sys.path.insert(0, str(_lab.resolve()))

import helper

import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances

from data.loader import load_iris

from helper import (
    graph_summary,
    load_moons_graph_data,
    plot_distance_matrix,
    plot_graph_and_matrix,
)
from lab02_3 import (
    apply_edge_weights,
    epsilon_graph,
    knn_graph,
)

pd.set_option("display.max_colwidth", 100)

## 3.1 Euclidean Distance

We first use Iris `petal length` and `petal width` from
`data/record/iris.csv`. `load_iris()` writes that file if it is missing.

In [ ]:
iris = load_iris()
X_iris = iris[[
    "petal length (cm)",
    "petal width (cm)",
]].to_numpy(dtype=float)
species = iris["species"].to_numpy()

iris[[
    "petal length (cm)",
    "petal width (cm)",
    "species",
]].head()

For two objects,

$$
d(O_i,O_j)
=
\sqrt{
\sum_{p=1}^{m}
(x_{ip}-x_{jp})^2
}.
$$

In [ ]:
i = 0
j = 1

distance_01 = np.sqrt(
    np.sum(
        (X_iris[i] - X_iris[j]) ** 2
    )
)

print(
    "d(O_0, O_1):",
    round(distance_01, 4),
)

Now compute the Euclidean distance for every pair of Iris objects.

In [ ]:
D_iris = pairwise_distances(
    X_iris,
    metric="euclidean",
)

print(
    "Distance matrix:",
    D_iris.shape,
)

pd.DataFrame(
    D_iris[:6, :6]
).round(3)

Each entry `D_iris[i, j]` is $d(O_i,O_j)$. The heatmap samples 12 objects per
species so the three groups are visible; a 150 × 150 table would hide that
block structure.

In [ ]:
#| fig-cap: "Iris Euclidean distance matrix, sampled by species"

plot_distance_matrix(
    D_iris,
    title="Iris Euclidean Distance",
    labels=species,
)

> #### 💡 Tip
> What value should appear on the diagonal of a distance matrix?  
> Zero. The distance from an object to itself is $d(O_i,O_i)=0$.

## 3.2 Constructing a Nearest Neighbor Graph

The graph is constructed in three steps:

1. create a node for each object,
2. connect nearby objects,
3. assign edge weights.

### Step 1. Create Nodes

Each Iris object becomes one node.

In [ ]:
print(
    "Number of nodes:",
    len(X_iris),
)

### Step 2. Create Edges

We first use a small distance matrix to see how each graph is constructed.

In [ ]:
D_small = np.array([
    [0.0, 0.2, 0.8],
    [0.2, 0.0, 0.5],
    [0.8, 0.5, 0.0],
])

D_small

#### $\epsilon$-Graph

Connect $O_i$ and $O_j$ when

$$
d(O_i,O_j)
\le
\epsilon.
$$

For example, with $\epsilon=0.4$:

In [ ]:
epsilon_small = 0.4

A_small_epsilon = (
    D_small
    <= epsilon_small
)

np.fill_diagonal(
    A_small_epsilon,
    False,
)

A_small_epsilon

`True` indicates that the two objects are connected.

> #### ❗ Important
> Implement `epsilon_graph()` in `lab02_3.py`.
> Use `epsilon = 0.25` and construct the Iris $\epsilon$-graph.
>
> **Hint:** Compare `D` with `epsilon`, then set the diagonal to `False`.
>
> ```{python}
>
> epsilon = 0.25
>
> A_epsilon = epsilon_graph(
>     D_iris,
>     epsilon,
> )
>
> graph_summary(
>     A_epsilon
> )
> ```

In [ ]:
#| fig-cap: "Iris epsilon graph and sampled adjacency matrix"

plot_graph_and_matrix(
    X_iris,
    A_epsilon,
    A_epsilon.astype(float),
    labels=species,
    graph_title=f"Iris ε-Graph (ε = {epsilon})",
    matrix_title="Adjacency (1 = edge)",
    xlabel="petal length (cm)",
    ylabel="petal width (cm)",
)

> #### 💡 Tip
> Why can different objects have different numbers of neighbors in an
> $\epsilon$-graph?  
> $\epsilon$ is a fixed radius. A point in a dense region has many neighbors
> inside that radius; a point in a sparse region has few.

#### kNN Graph

If $O_j$ is among the $K$ nearest neighbors of $O_i$, add

$$
O_i
\rightarrow
O_j.
$$

For the same distance matrix, use $K=1$:

In [ ]:
K_small = 1
n_small = D_small.shape[0]

A_small_knn_directed = np.zeros(
    (n_small, n_small),
    dtype=bool,
)

for i in range(n_small):
    distances = D_small[i].copy()
    distances[i] = np.inf

    neighbors = (
        np.argsort(distances)[:K_small]
    )

    A_small_knn_directed[
        i,
        neighbors,
    ] = True

A_small_knn_directed

Each row shows the nearest-neighbor edge selected from one object.
The direction is often ignored.

In [ ]:
A_small_knn = (
    A_small_knn_directed
    | A_small_knn_directed.T
)

A_small_knn

> #### ❗ Important
> Implement `knn_graph()` in `lab02_3.py`.
> Apply the same kNN rule to every Iris object using `K = 4`, then ignore
> direction.
>
> **Hint:** For each row, set the self-distance to `inf`, sort the distances,
> and keep the first `K` indices. Symmetrize with a transpose.
>
> ```{python}
>
> K = 4
>
> A_knn = knn_graph(
>     D_iris,
>     k=K,
> )
>
> graph_summary(
>     A_knn
> )
> ```

In [ ]:
#| fig-cap: "Iris kNN graph and sampled adjacency matrix"

plot_graph_and_matrix(
    X_iris,
    A_knn,
    A_knn.astype(float),
    labels=species,
    graph_title=f"Iris kNN Graph (K = {K})",
    matrix_title="Adjacency (1 = edge)",
    xlabel="petal length (cm)",
    ylabel="petal width (cm)",
)

> #### 💡 Tip
> Why can a node have more than $K$ neighbors after direction is
> ignored?  
> kNN is directed. If $A$ lists $B$ as a neighbor, the undirected graph keeps
> the edge even when $A$ is not among $B$'s $K$ nearest. Extra incoming edges
> raise the degree above $K$.

Compare the two Iris summaries. The $\epsilon$-graph can have
`isolates > 0` and a large `max_degree` in the dense setosa cluster. The
kNN graph has `isolates = 0` and `min_degree` at least $K$, while
`max_degree` can still exceed $K$. Isolates are circled on the $\epsilon$-graph.

### Step 3. Heat-Kernel Weights

The ε-graph and kNN graph only decide **which pairs are connected**. They do
not yet say how strong those edges are.

Heat-kernel weights reuse the Euclidean matrix `D_iris`. They do not compute
a new distance. They map each already-computed distance to a similarity in
$(0,1]$:

$$
w_{ij}
=
\exp
\left(
-\frac{d(O_i,O_j)^2}
{t^2}
\right).
$$

Close pairs stay near 1, distant pairs go toward 0. The bandwidth $t$
controls how fast that happens: smaller $t$ makes similarity decay faster.
We set $t=\epsilon$, so a pair exactly at the ε-graph threshold has weight
$e^{-1}\approx 0.37$ and nearer pairs are thicker and darker. Then
`apply_edge_weights()` keeps $w_{ij}$ only on the ε or kNN edges and puts 0
elsewhere.

In [ ]:
t = epsilon

W = np.exp(
    -(D_iris ** 2)
    / (t ** 2)
)

A small sample, two flowers from each species, shows the decay. Nearby
setosa pairs stay close to 1; a setosa-virginica pair is much smaller.

In [ ]:
sample_idx = [0, 1, 50, 51, 100, 101]
sample_names = [
    f"{i} ({species[i]})"
    for i in sample_idx
]

print("Euclidean distances")
display(
    pd.DataFrame(
        D_iris[np.ix_(sample_idx, sample_idx)],
        index=sample_names,
        columns=sample_names,
    ).round(3)
)

print("Heat-kernel similarities, t =", t)
display(
    pd.DataFrame(
        W[np.ix_(sample_idx, sample_idx)],
        index=sample_names,
        columns=sample_names,
    ).round(3)
)

> #### ❗ Important
> Implement `apply_edge_weights()` in `lab02_3.py`.
> Keep heat-kernel weights only for the edges selected by the $\epsilon$-graph
> and kNN graph.
>
> **Hint:** Use `np.where()` with the adjacency matrix and `W`.
>
> ```{python}
>
> W_epsilon = apply_edge_weights(
>     W,
>     A_epsilon,
> )
>
> W_knn = apply_edge_weights(
>     W,
>     A_knn,
> )
>
> pd.DataFrame(
>     W_epsilon[:6, :6]
> ).round(3)
> ```

In [ ]:
#| fig-cap: "Iris weighted epsilon graph and sampled similarity matrix"

plot_graph_and_matrix(
    X_iris,
    A_epsilon,
    W_epsilon,
    labels=species,
    weights=W_epsilon,
    graph_title="Iris Weighted ε-Graph",
    matrix_title="Heat-kernel weights on ε-edges",
    xlabel="petal length (cm)",
    ylabel="petal width (cm)",
)

In [ ]:
#| fig-cap: "Iris weighted kNN graph and sampled similarity matrix"

plot_graph_and_matrix(
    X_iris,
    A_knn,
    W_knn,
    labels=species,
    weights=W_knn,
    graph_title="Iris Weighted kNN Graph",
    matrix_title="Heat-kernel weights on kNN edges",
    xlabel="petal length (cm)",
    ylabel="petal width (cm)",
)

> #### 💡 Tip
> What happens to $w_{ij}$ as the Euclidean distance increases?  
> The heat-kernel weight decays toward 0. Distant pairs contribute almost no
> similarity.

## 3.3 Similarity Graphs on Moons

We now apply the **same functions** to the Moons dataset. Do not reimplement
the graph rules here. Reuse `epsilon_graph()`, `knn_graph()`, and
`apply_edge_weights()` from `lab02_3.py`.

In [ ]:
X_moons, y_moons = (
    load_moons_graph_data()
)

D_moons = pairwise_distances(
    X_moons,
    metric="euclidean",
)

pd.DataFrame(
    D_moons[:6, :6]
).round(3)

The labels `y_moons` are used only for visualization.

The construction is the same three steps as Iris: Euclidean distances, then
ε or kNN edges, then heat-kernel weights on those edges. Moons uses a
smaller $\epsilon$ than Iris so sparse parts of the crescent break into
isolates, while kNN still connects every point.

### $\epsilon$-Graph

In [ ]:
epsilon_moons = 0.12

A_moons_epsilon = epsilon_graph(
    D_moons,
    epsilon_moons,
)

graph_summary(
    A_moons_epsilon
)

In [ ]:
#| fig-cap: "Moons epsilon graph and sampled adjacency matrix"

plot_graph_and_matrix(
    X_moons,
    A_moons_epsilon,
    A_moons_epsilon.astype(float),
    labels=y_moons,
    graph_title=f"Moons ε-Graph (ε = {epsilon_moons})",
    matrix_title="Adjacency (1 = edge)",
)

### kNN Graph

In [ ]:
K_moons = 4

A_moons_knn = knn_graph(
    D_moons,
    k=K_moons,
)

graph_summary(
    A_moons_knn
)

In [ ]:
#| fig-cap: "Moons kNN graph and sampled adjacency matrix"

plot_graph_and_matrix(
    X_moons,
    A_moons_knn,
    A_moons_knn.astype(float),
    labels=y_moons,
    graph_title=f"Moons kNN Graph (K = {K_moons})",
    matrix_title="Adjacency (1 = edge)",
)

### Heat-Kernel Weights

`D_moons` was already used to choose the edges. Heat-kernel weights map those
same Euclidean distances to similarities. They do not add new neighbors.
With $t=\epsilon$, a pair at the ε-graph threshold gets weight $e^{-1}\approx 0.37$,
and farther pairs (kept only by kNN) get even smaller weights.

$$
w_{ij}
=
\exp
\left(
-\frac{d(O_i,O_j)^2}
{t^2}
\right).
$$

In [ ]:
t_moons = epsilon_moons

W_moons = np.exp(
    -(D_moons ** 2)
    / (t_moons ** 2)
)

W_moons_epsilon = apply_edge_weights(
    W_moons,
    A_moons_epsilon,
)

W_moons_knn = apply_edge_weights(
    W_moons,
    A_moons_knn,
)

In [ ]:
#| fig-cap: "Weighted Moons epsilon graph and sampled similarity matrix"

plot_graph_and_matrix(
    X_moons,
    A_moons_epsilon,
    W_moons_epsilon,
    labels=y_moons,
    weights=W_moons_epsilon,
    graph_title="Moons Weighted ε-Graph",
    matrix_title="Heat-kernel weights on ε-edges",
)

In [ ]:
#| fig-cap: "Weighted Moons kNN graph and sampled similarity matrix"

plot_graph_and_matrix(
    X_moons,
    A_moons_knn,
    W_moons_knn,
    labels=y_moons,
    weights=W_moons_knn,
    graph_title="Moons Weighted kNN Graph",
    matrix_title="Heat-kernel weights on kNN edges",
)

> #### 💡 Tip
> How do the $\epsilon$-graph and kNN graph differ in selecting
> neighbors?  
> The $\epsilon$-graph uses a distance threshold, so the number of neighbors
> varies and sparse points can become isolates. The kNN graph uses a neighbor
> count, so each node has $K$ outgoing neighbors and isolates disappear.

> #### 💡 Tip
> When density varies across the data, which graph is usually easier to use
> in practice, and why?  
> A kNN graph. One global $\epsilon$ is too large in dense regions and too
> small in sparse regions, so the same radius cannot fit both. kNN asks for
> $K$ neighbors everywhere, so it does not need a single distance cutoff.